# Sentiment Analysis - Mobile Phone Reviews
## Production-Ready NLP Pipeline

In [ ]:
# ===== 1. IMPORT LIBRARIES =====
import pandas as pd
import numpy as np
import re
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from textblob import TextBlob

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab' quiet=True)
nltk.download('stopwords' quiet=True)
nltk.download('wordnet' quiet=True)

True

In [2]:
# ===== 2. LOAD DATA =====
df = pd.read_excel("dataset.xlsx")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1440, 3)


,title,rating,body
0,Horrible product,1,Very disappointed with the overall performance...
1,Camera quality is not like 48 megapixel,3,Camera quality is low
2,Overall,4,"Got the mobile on the launch date,Battery must..."
3,A big no from me,1,1. It doesn't work with 5.0GHz WiFi frequency....
4,Put your money somewhere else,1,"Not worth buying....faulty software, poor disp..."


In [3]:
# ===== 3. COMBINE TITLE AND BODY =====
df['text'] = df['title'] + " " + df['body']
print(df[['text', 'rating']].head())

                                                text  rating
0  Horrible product Very disappointed with the ov...       1
1  Camera quality is not like 48 megapixel Camera...       3
2  Overall Got the mobile on the launch date,Batt...       4
3  A big no from me 1. It doesn't work with 5.0GH...       1
4  Put your money somewhere else Not worth buying...       1


In [4]:
# ===== 4. DEFINE PREPROCESSING FUNCTION (REUSABLE) =====

# Initialize lemmatizer and stopwords ONCE (outside the function for efficiency)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Clean and preprocess a single text string.
    This function is used BOTH during training and prediction.
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+", "", text)
    
    # Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

# Test the function
sample_text = "This product is AMAZING!!! I love it 😊 http://example.com"
print(f"Original: {sample_text}")
print(f"Cleaned: {preprocess_text(sample_text)}")

Original: This product is AMAZING!!! I love it 😊 http://example.com


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\HI/nltk_data'
    - 'c:\\Users\\HI\\anaconda3\\nltk_data'
    - 'c:\\Users\\HI\\anaconda3\\share\\nltk_data'
    - 'c:\\Users\\HI\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\HI\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [ ]:
# ===== 5. APPLY PREPROCESSING TO ALL DATA =====
df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text']].head()

In [ ]:
# ===== 6. CREATE SENTIMENT LABELS =====

# Calculate polarity scores
df['polarity'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.polarity)

# Define label function
def get_sentiment_label(polarity_score):
    if polarity_score > 0.2:
        return "Positive"
    elif polarity_score < -0.2:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['polarity'].apply(get_sentiment_label)

# Check class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# Visualize
df['sentiment'].value_counts().plot(kind='bar', color=['red', 'gray', 'green'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
# ===== 7. PREPARE FEATURES AND LABELS =====
X = df['clean_text']
y = df['sentiment']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# ===== 8. TRAIN-TEST SPLIT =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nTraining set distribution:\n{y_train.value_counts()}")

In [ ]:
# ===== 9. BUILD ML PIPELINE =====
# This pipeline will handle EVERYTHING: preprocessing is already done, just vectorization + model

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=7000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Handle class imbalance
        random_state=42
    ))
])

print("Pipeline created successfully!")
print(pipeline)

In [ ]:
# ===== 10. TRAIN MODEL =====
print("Training model...")
pipeline.fit(X_train, y_train)
print("✓ Training complete!")

In [ ]:
# ===== 11. EVALUATE MODEL =====

# Predictions
y_pred = pipeline.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*50}")
print(f"TEST ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}\n")

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Negative', 'Neutral', 'Positive'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - Accuracy: {accuracy:.2%}")
plt.show()

In [ ]:
# ===== 12. CROSS-VALIDATION SCORE =====
print("Running 5-Fold Cross-Validation...")
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"\nCross-Validation Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
# ===== 13. TEST ON RAW INPUT (SIMULATING WEB APP) =====

def predict_sentiment(raw_text):
    """
    Predict sentiment from RAW user input.
    Preprocessing happens INSIDE this function.
    """
    # Step 1: Preprocess the raw text
    cleaned = preprocess_text(raw_text)
    
    # Step 2: Predict using pipeline (which does TF-IDF internally)
    prediction = pipeline.predict([cleaned])[0]
    probabilities = pipeline.predict_proba([cleaned])[0]
    
    return prediction, probabilities

# Test cases
test_reviews = [
    "This phone is AMAZING!!! Best purchase ever 😊",
    "Terrible product, waste of money",
    "It's okay, nothing special",
    "Battery backup is unimaginable. Great phone for the price!",
    "Camera quality is very poor, disappointed"
]

print("\n" + "="*70)
print("TESTING PREDICTIONS ON RAW INPUT (Like Web App Will Do)")
print("="*70 + "\n")

for review in test_reviews:
    sentiment, probs = predict_sentiment(review)
    confidence = max(probs) * 100
    
    print(f"Review: {review}")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.1f}%)")
    print(f"Probabilities: Neg={probs[0]:.2f}, Neu={probs[1]:.2f}, Pos={probs[2]:.2f}")
    print("-" * 70)

In [ ]:
# ===== 14. SAVE EVERYTHING FOR DEPLOYMENT =====

# Save the complete pipeline
joblib.dump(pipeline, 'sentiment_pipeline.pkl')
print("✓ Pipeline saved as: sentiment_pipeline.pkl")

# Save the preprocessing function separately (for documentation)
joblib.dump(preprocess_text, 'preprocess_function.pkl')
print("✓ Preprocessing function saved as: preprocess_function.pkl")

print("\n" + "="*70)
print("MODEL TRAINING COMPLETE!")
print("Files saved:")
print("  1. sentiment_pipeline.pkl  (TF-IDF + Logistic Regression)")
print("  2. preprocess_function.pkl (Text cleaning function)")
print("="*70)